# 00 - Ingest Raw CSV

Lee los CSV cargados en `Files/raw` y crea tablas Delta `raw.*` en el Lakehouse adjunto.

In [ ]:
# Fabric notebook: 00_ingest_raw
# Parameters:
#   RAW_FILES_PATH = "Files/raw"
#   RAW_SCHEMA = "raw"

from pyspark.sql.functions import current_timestamp, input_file_name, lit

raw_files_path = "Files/raw"
raw_schema = "raw"
tables = [
    "crm",
    "productos",
    "tiendas",
    "pedidos",
    "lineas_pedido",
    "ventas_pos",
    "pagos_tienda",
    "devoluciones_online",
    "devoluciones_tienda",
    "stock_diario",
    "fact_transacciones",
]

spark.sql(f"CREATE SCHEMA IF NOT EXISTS {raw_schema}")

for table in tables:
    path = f"{raw_files_path}/{table}.csv"
    df = (
        spark.read.option("header", "true")
        .option("multiLine", "true")
        .option("escape", '"')
        .csv(path)
        .withColumn("_source_file", input_file_name())
        .withColumn("_ingested_at", current_timestamp())
        .withColumn("_data_zone", lit("raw"))
    )
    df.write.mode("overwrite").format("delta").saveAsTable(f"{raw_schema}.{table}")

    # Compatibility view for existing demo queries/notebooks that still reference bronze_*.
    spark.sql(f"CREATE OR REPLACE VIEW bronze_{table} AS SELECT * FROM {raw_schema}.{table}")
    print(f"{raw_schema}.{table}: {df.count()} rows")

display(spark.sql("SHOW TABLES"))
